In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
   전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}

</style>
"""))


<b><font color="red" size="6">ch15. 데이터베이스 연동</font></b>
# 1절. SQLite 데이터 베이스 연결
- SQLite 데이터베이스는 별도의 DBMS없이 SQL을 이용해서 DB 엑세스할 수 있도록 만든 간단한 디스크 기반 DB 제공
- C 라이브러리
- SQLite는 프로토타입을 만들 때 사용
- 프로젝트단계 : 분석 -> 설계 -> 구현 -> 테스트 -> 고객에게 배포 -> 유지보수
- 프로토타입(SQLite : 시제품용 데이터베이스) 시제품(구현 후 반양산직전) 완제품(Oracle, MySQL, MariaDB, PostgreSQL, MSSQL, 아마존auroraDB, ...)
- DB Browser for SQLite에서 "DB Browser for SQLite - .zip (no installer) for 64-bit Windows" 다운로드후 압축 풀기

In [16]:
import sqlite3
sqlite3.sqlite_version

'3.40.1'

In [2]:
import pandas as pd
pd.__version__

'1.5.3'

## 1.2 데이터베이스 연결
- 데이터베이스 연결객체 -> 커서객체(SQL전송 및 결과 받는 객체) -> 원하는 로직 수행 -> 커서객체 해제 -> DB연결객체 해제(close)
- SQLite로 DB 연결객체 생성시, DB파일이 있으면 연결, DB파일이 없으면 빈 DB파일 생성

In [3]:
# DB연결 (여기서 에러가 날 경우 VC_readist.x64.exe 설치)
conn = sqlite3.connect('data/ch15_example.db')
conn

In [4]:
# 커서 객체 생성 : 커서는 SQL문 실행시키고, 결과를 받는 객체
cursor = conn.cursor()
cursor

In [9]:
cursor.execute('''
    CREATE TABLE MEMBER (
            NAME TEXT,
            AGE INT,
            EMAIL TEXT
    )
''')

In [8]:
cursor.execute('DROP TABLE MEMBER')

In [18]:
sql = 'INSERT INTO MEMBER VALUES (\'홍길동\', 25, \'h@h.com\')'
cursor.execute(sql) # sql 전송
print('insert, update, delete문의 수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신길동', 30, 's@h.com')"
cursor.execute(sql)
print('수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신림동', 35, 'sil@h.com')"
cursor.execute(sql)
print('수행 결과 행수 :', cursor.rowcount)

insert, update, delete문의 수행 결과 행수 : 1
수행 결과 행수 : 1
수행 결과 행수 : 1


In [19]:
conn.commit() # 反.conn.rollback()DML에서만 commit이나 rollback

In [8]:
cursor.execute('''
    SELECT * 
    FROM MEMBER 
    ORDER BY AGE
    ''') # SELECT sql문 전송

In [9]:
# INSERT, UPDATE, DELETE문 실행결과 : cursor.rowcount
# SELECT문 실행결과를 받는 함수들
    # fetchone() : 결과를 한 행씩 받을 때 (튜플)
    # fetchall()  : 결과를 모두 받을 때 (튜플 list)
    # fetchmant(n) : 결과를 n행 받을 때 (튜플 list)
cursor.fetchall()

[('홍길동', 25, 'h@h.com'), ('신길동', 30, 's@h.com'), ('신림동', 35, 'sil@h.com')]

In [10]:
cursor.fetchall() # 한번 소요된 cursor 객체는 다시 fetch 할 수 없음.

[]

In [11]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    member = cursor.fetchone() # SQL문 수행결과를 한줄 가져오기
    if member is None:
        break
    members.append({'name':member[0],'age':member[1], 'email':member[2]})
members

[{'name': '홍길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '신길동', 'age': 30, 'email': 's@h.com'},
 {'name': '신림동', 'age': 35, 'email': 'sil@h.com'}]

In [12]:
class Member:
    'Member 테이블의 내용을 받은 객체 타입'
    def __init__(self, name, age, email):
        self.name = name
        self.age = age
        self.email = email
    def __str__(self):
        return "{}\t{}\t{}".format(self.name, self.age, self.email)
m = Member('홍길동', 25, 'h@h.com')
print(m)

홍길동	25	h@h.com


In [13]:
# 한줄씩 읽어서 객체list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    dbmember = cursor.fetchone()
    if dbmember is None:
        break
    member = Member(*dbmember)
    members.append(member)
for mem in members:
    print(mem)

홍길동	25	h@h.com
신길동	30	s@h.com
신림동	35	sil@h.com


In [11]:
# 최상위 n행 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchmany(2)
members

[('홍길동', 25, 'h@h.com'), ('신길동', 30, 's@h.com')]

In [12]:
cursor.close()
conn.close()

## 1.3 SQL구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우도 있음)
- named(추천)

In [21]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('홍길동', '신길동')")
cursor.fetchall()

[('홍길동', 25, 'h@h.com'), ('신길동', 30, 's@h.com')]

In [22]:
# 파라미터 사용하기 : qmark 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (:name1, :name2)", {'name1':name1, 'name2':name2})
cursor.fetchall()

검색할 이름1 :김길동
검색할 이름2 :홍길동


[('홍길동', 25, 'h@h.com')]

In [23]:
# 파라미터 사용하기 : named 방법
spl = "INSERT INTO MEMBER VALUES (:name, :age, :email)"
name = input('회원가입할 이름은?')
try:
    age = int(input('나이는?(꼭 숫자로)'))
except:
    print('유효하지 않는 나이를 입력할 경우 1세로 초기화됩니다.')
    age = 1
email = input('이메일은?')

cursor.execute("INSERT INTO MEMBER VALUES (:name, :age, :email)",{'name':name, 'age':age, 'email':email}) # sql 전송
conn.commit()

print('수행 결과 행수 :', cursor.rowcount)

회원가입할 이름은?김낄뚱
나이는?(꼭 숫자로)이팔청춘
유효하지 않는 나이를 입력할 경우 1세로 초기화됩니다.
이메일은?L@L.COM
수행 결과 행수 : 1


In [24]:
cursor.close()
conn.close()

# 2절. 오라클 데이터베이스 연결
- pip install cx_oracle(11g까지), pip install cx_oracle(명령 프롬포트 실행)

In [25]:
import cx_Oracle
cx_Oracle.__version__

'8.3.0'

In [27]:
# conn 얻어오는 방법1
oracle_dsn = cx_Oracle.makedsn(host="localhost", port=1521, sid='xe')
# conn = cx_Oracle.connect(user='scott', password='tiger', dsn=oracle_dsn)
conn = cs_Oracle.connect('scott', 'tiger', oracle_dsn)
conn.close()

In [29]:
# conn 얻어오는 방법2
#conn = cx_Oracle.connect(user='scott', password='tiger', dsn='localhost:1521/xe')
conn = cx_Oracle.connect('scott', 'tiger', 'localhost:1521/xe')
conn

<cx_Oracle.Connection to scott@localhost:1521/xe>

In [30]:
# cursor 객체 생성하고 sql문 전송&결과 받기
cursor = conn.cursor()
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, HIREDATE, SAL, COMM, DEPTNO FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()

In [31]:
for emp in emps:
    print(emp)

(7369, 'SMITH', 'CLERK', 7902, datetime.datetime(1980, 12, 17, 0, 0), 800.0, None, 20)
(7499, 'ALLEN', 'SALESMAN', 7698, datetime.datetime(1981, 2, 20, 0, 0), 1600.0, 300.0, 30)
(7521, 'WARD', 'SALESMAN', 7698, datetime.datetime(1981, 2, 22, 0, 0), 1250.0, 500.0, 30)
(7566, 'JONES', 'MANAGER', 7839, datetime.datetime(1981, 4, 2, 0, 0), 2975.0, None, 20)
(7654, 'MARTIN', 'SALESMAN', 7698, datetime.datetime(1981, 9, 28, 0, 0), 1250.0, 1400.0, 30)
(7698, 'BLAKE', 'MANAGER', 7839, datetime.datetime(1981, 5, 1, 0, 0), 2850.0, None, 30)
(7782, 'CLARK', 'MANAGER', 7839, datetime.datetime(1981, 6, 9, 0, 0), 2450.0, None, 10)
(7788, 'SCOTT', 'ANALYST', 7566, datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20)
(7839, 'KING', 'PRESIDENT', None, datetime.datetime(1981, 11, 17, 0, 0), 5000.0, None, 10)
(7844, 'TURNER', 'SALESMAN', 7698, datetime.datetime(1981, 9, 8, 0, 0), 1500.0, 0.0, 30)
(7876, 'ADAMS', 'CLERK', 7788, datetime.datetime(1983, 1, 12, 0, 0), 1100.0, None, 20)
(7900, 'JAMES', 'CL

In [32]:
cursor.description

[('NO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 0),
 ('ENAME', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 10, 10, None, None, 1),
 ('JOB', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 9, 9, None, None, 1),
 ('MGR', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 1),
 ('HIREDATE', <cx_Oracle.DbType DB_TYPE_DATE>, 23, None, None, None, 1),
 ('SAL', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('COMM', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('DEPTNO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 3, None, 2, 0, 1)]

In [35]:
import pandas as pd
emp_df = pd.DataFrame(emps)
emp_df.head()

,0,1,2,3,4,5,6,7
0,7369,SMITH,CLERK,7902.0,1980-12-17,800.0,NaN,20
1,7499,ALLEN,SALESMAN,7698.0,1981-02-20,1600.0,300.0,30
2,7521,WARD,SALESMAN,7698.0,1981-02-22,1250.0,500.0,30
3,7566,JONES,MANAGER,7839.0,1981-04-02,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,7698.0,1981-09-28,1250.0,1400.0,30


In [37]:
[descipt[0] for descipt in cursor.description]

['NO', 'ENAME', 'JOB', 'MGR', 'HIREDATE', 'SAL', 'COMM', 'DEPTNO']

In [38]:
import pandas as pd
emp_df = pd.DataFrame(emps, columns=[descipt[0] for descipt in cursor.description])
emp_df

,NO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902.0,1980-12-17,800.0,NaN,20
1,7499,ALLEN,SALESMAN,7698.0,1981-02-20,1600.0,300.0,30
2,7521,WARD,SALESMAN,7698.0,1981-02-22,1250.0,500.0,30
3,7566,JONES,MANAGER,7839.0,1981-04-02,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,7698.0,1981-09-28,1250.0,1400.0,30
5,7698,BLAKE,MANAGER,7839.0,1981-05-01,2850.0,NaN,30
6,7782,CLARK,MANAGER,7839.0,1981-06-09,2450.0,NaN,10
7,7788,SCOTT,ANALYST,7566.0,1982-12-09,3000.0,NaN,20
8,7839,KING,PRESIDENT,NaN,1981-11-17,5000.0,NaN,10
9,7844,TURNER,SALESMAN,7698.0,1981-09-08,1500.0,0.0,30


In [48]:
# 사용자로부터 검색할 이름을 받아 해당 데이터 출력
sql = "SELECT * FROM EMP WHERE ENAME=(:ename)"
ename = input('검색할 이름? ').upper()
cursor.execute(sql, {'ename':ename})
emp = cursor.fetchone() # 결과가 있으면 해당 데이터를 튜플로, 결과가 없으면 None으로 
if emp:
    columns = [descript[0] for descript in cursor.description]
    df = pd.DataFrame([emp], columns=columns)
    display(df)
else:
    print('해당 이름이 없습니다')

검색할 이름? scott


,EMPNO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7788,SCOTT,ANALYST,7566,1982-12-09,3000.0,None,20


In [50]:
for col, data in zip(columns, emp):
    print("{}:{}".format(col, data if data is not None else '-'))

EMPNO:7788
ENAME:SCOTT
JOB:ANALYST
MGR:7566
HIREDATE:1982-12-09 00:00:00
SAL:3000.0
COMM:-
DEPTNO:20


In [51]:
cursor.close()
conn.close()

# 3절. MySQL 연결

| 라이브러리 | 특징 |
| : -- | : -- |
| **mysql-connector-python** | MySQL 공식 커넥터. python으로 구현 |
| **PyMySQL** | 커뮤니티에서 만든 서드파트 라이브러리(경량). python으로 구현. 널리 쓰임 |

- pip install pyMySQL

In [5]:
%pip install pyMySQL

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pymysql
conn = pymysql.connect(
    host = '127.0.0.1', # 또는 'Localhost'
  #port = 3306,
    user = 'root',
    password = '6561',
    database = 'dev_db',
    charset = 'utf8mb4', # 이모지 추가 및 한글깨짐 방지
    autocommit = True  # 자동 커밋
)
cursor = conn.cursor()
sql = 'select * from person'
cursor.execute(sql) # sql 전송하고 결과 받기
person = cursor.fetchall()
print(person)
cursor.close()
conn.close()

((1001, 'bill', 'president', None, datetime.date(1989, 1, 10), Decimal('7000'), None, 10), (1111, 'smith', 'manager', 1001, datetime.date(1990, 12, 17), Decimal('1000'), None, 10), (1112, 'ally', 'salesman', 1116, datetime.date(1991, 2, 20), Decimal('1600'), Decimal('500'), 30), (1113, 'word', 'salesman', 1116, datetime.date(1992, 2, 24), Decimal('1450'), Decimal('300'), 30), (1114, 'james', 'manager', 1001, datetime.date(1990, 4, 12), Decimal('3975'), None, 20), (1116, 'johnson', 'manager', 1001, datetime.date(1991, 5, 1), Decimal('3550'), None, 30), (1118, 'martin', 'analyst', 1111, datetime.date(1991, 9, 9), Decimal('3450'), None, 10), (1121, 'kim', 'clerk', 1114, datetime.date(1990, 12, 8), Decimal('4000'), None, 20), (1123, 'lee', 'salesman', 1116, datetime.date(1991, 9, 23), Decimal('1200'), Decimal('0'), 30), (1226, 'park', 'analyst', 1111, datetime.date(1990, 1, 3), Decimal('2500'), None, 10))


In [16]:
import pandas as pd
import numpy as np
df = pd.DataFrame(
    person, columns=[descript[0] for descript in cursor.description]
)
#df.fillna(np.nan)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pno       10 non-null     int64  
 1   pname     10 non-null     object 
 2   job       10 non-null     object 
 3   manager   9 non-null      float64
 4   hiredate  10 non-null     object 
 5   sal       10 non-null     object 
 6   comm      3 non-null      object 
 7   dno       10 non-null     int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 768.0+ bytes


In [18]:
df.isna().sum()

pno         0
pname       0
job         0
manager     1
hiredate    0
sal         0
comm        7
dno         0
dtype: int64

In [19]:
df['hiredate'] = df['hiredate'].astype('datetime64[ns]')

In [20]:
df.astype({'sal':'float64', 'comm':np.float64})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   pno       10 non-null     int64         
 1   pname     10 non-null     object        
 2   job       10 non-null     object        
 3   manager   9 non-null      float64       
 4   hiredate  10 non-null     datetime64[ns]
 5   sal       10 non-null     object        
 6   comm      3 non-null      object        
 7   dno       10 non-null     int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 768.0+ bytes


# 4절. 연습문제
## oracle 연동
- 회원가입 | 전체조회 | 이름찾기 | 메일삭제 | csv내보내기 | 종료
### 0. 처음실행

In [55]:
def load_conn():
    global conn # 변수를 전역변수로 쓰겠다
    import cx_Oracle
    conn = cx_Oracle.connect("scott", "tiger", "localhost:1521/xe")
load_conn()

### 1. 회원가입

In [62]:
def fn1_insert_member():
    '사용자로부터 이름, 전화, 이메일, 나이, 등급(1~5)을 입력받아 DB에 insert한다.'
    name = input('이름 : ')
    phone = input('전화 : ')
    email = input('메일 : ')
    try:
        age = int(input('나이 : '))
        if age<0 : age=0
    except:
        print('유효하지 않은 나이 입력시 나이는 0으로 초기화됩니다.')
        age = 0
    try:
        grade = int(input('등급(1~5) : '))
        if grade < 1:
            grade = 1
        elif grade > 5:
               grade = 5
    except:
        print('유효하지 않은 등급 입력시 1로 초기화')
        grade = 1
    # SQL 전송 및 결과 받기
    cursor = conn.cursor()
    sql = "INSERT INTO MEMBER VALUES (:name, :phone, :email, :age, :grade)"
    cursor.execute(sql, 
                   {'name':name, 'phone':phone, 'email':email, 'age':age, 'grade':grade} # sql 전송 및 결과 받기
                  )
    if cursor.rowcount:
        conn.commit()
        print(name + '님 회원가입 완료!')
        
    cursor.close()
    
# fn1_insert_member()

### 2. 전체조회

In [63]:
def fn2_display_members():
    'member 테이블의 내용을 데이터프레임으로 display'
    import pandas as pd
    cursor = conn.cursor()
    sql = """
    SELECT NAME, PHONE, EMAIL, AGE, GRADE 
    FROM MEMBER
    ORDER BY AGE
    """
    cursor.execute(sql)
    members = cursor.fetchall() # 데이터가 존재하면 튜플 리스트, 데이터가 없다면 빈 리스트
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(
            members, columns=columns
        )
        display(df)
    else:
        print('입력된 회원이 없습니다')
    cursor.close()

# fn2_display_members()

### 3. 이름으로 조회
- SELECT * FROM MEMBER WHERE NAME = '홍길동';

In [64]:
def fn3_search_name():
    name = input('검색할 이름: ')
    sql = """
    SELECT *
    FROM MEMBER
    WHERE NAME = :name
    """
    cursor = conn.cursor()
    cursor.execute(sql, {'name':name})
    name_data = cursor.fetchall()
    if name_data:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(
            name_data, columns=columns
        )
        display(df)
    else:
        print('해당 이름은 없는 회원입니다')
    
    print(name_data)
    cursor.close()
    
# fn3_search_name()

### 4. 메일로 삭제
- DELETE FROM MEMBER WHERE UPPER (EMAIL)=UPPER('abc@h.com');
- commit;

In [65]:
def fn4_delete_member():
    email = input('삭제할 회원의 email 조회 후 삭제: ')
    cursor = conn.cursor()
    sql = '''
    SELECT NAME
    FROM MEMBER
    WHERE LOWER (EMAIL) = LOWER(:email)
    '''
    cursor.execute(sql, {'email':email})
    
    members = cursor.fetchall() # [(홍길동, 010), (ghd)]
    if members:
        names = [member[0] for member in members] # 삭제할 회원이름들
        sql = "DELETE FROM MEMBER WHERE UPPER (EMAIL)=LOWER(:email)"
        cursor.execute(sql, {'email':email})
        conn.commit()
        print(f"{names}님 회원 삭제 완료")
    else:
        print('해당 이메일의 회원은 없습니다')
    cursor.close()

# fn4_delete_member()

### 5. csv 내보내기
- data/ch15_member.csv로 회원정보 내보내기

In [66]:
def fn5_save_csv():
    import pandas as pd
    cursor = conn.cursor()
    sql = """SELECT NAME, PHONE, EMAIL, AGE, GRADE 
                FROM MEMBER 
                ORDER BY AGE"""
    cursor.execute(sql)
    members = cursor.fetchall() # 데이터가 있으면 튜플list, 데이터가 없으면 빈 list
    columns = [descript[0] for descript in cursor.description]
    df = pd.DataFrame(members, columns=columns)
    df.to_csv('data/ch15_member.csv', index=False)
    print('csv파일 백업 완료')
    cursor.close()
# fn5_save_csv()

### 6. 기능들 합치기

In [71]:
def main():
    while True:
        menu = input("1. 회원입력 | 2. 전체조회 | 3. 이름찾기 | 4. 메일삭제 | 5. csv백업 | 6. 종료" )
        if menu=='1':
            fn1_insert_member()
        elif menu=='2':
            fn2_display_members()
        elif menu=='3':
            fn3_search_name()
        elif menu=='4':
            fn4_delete_member()
        elif menu=='5':
            fn5_save_csv()
        elif menu=='9':
            conn.close()
            break
        else:
            print('유효한 메뉴 번호를 입력해주세요.')
if __name__=='__main__':
    import cx_Oracle
    conn = cx_Oracle.connect("scott", "tiger", "localhost:1521/xe")
#     import sqlite3
#     conn = sqlite3.connect('data/ch15_member.db')
    main()

1. 회원입력 | 2. 전체조회 | 3. 이름찾기 | 4. 메일삭제 | 5. csv백업 | 6. 종료2


,NAME,PHONE,EMAIL,AGE,GRADE
0,곽한구,112,seoulchoung112@naver.com,10,1
1,홍길동,010-9999-9999,h@h.com,25,1


1. 회원입력 | 2. 전체조회 | 3. 이름찾기 | 4. 메일삭제 | 5. csv백업 | 6. 종료9
